# Program 9 – Softmax Classifier on MNIST

**Objective:** Implement a classifier using softmax activation for 10-class digit recognition.

**Dataset:** MNIST

**Key tasks**
1. Normalize data.
2. Use softmax in the final output layer.
3. Compile with `categorical_crossentropy`.
4. Evaluate accuracy and confusion matrix.

## Theory (for viva)

A **softmax classifier** (multinomial logistic regression) is a single linear layer followed by softmax:
$$z = Wx + b, \qquad P(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{10} e^{z_j}}$$
- $x$: 784 pixels, $W$: 784 × 10 weights, $b$: 10 biases → **7,850 parameters**.
- Prediction = class with the highest probability ($\arg\max$).
- **Loss:** categorical cross-entropy $L = -\sum_k y_k \log p_k$ with one-hot labels $y$.
- **Normalization:** scaling pixels from [0, 255] to [0, 1] keeps inputs small → stable, faster gradient descent.
- **One-hot encoding:** label 3 → `[0,0,0,1,0,0,0,0,0,0]` (needed for `categorical_crossentropy`; `sparse_categorical_crossentropy` takes plain integers).
- **Confusion matrix:** row = true class, column = predicted class; diagonal = correct predictions.

## Step 1 – Import libraries

In [ ]:
import keras  # Keras 3 API (runs on TensorFlow backend)
from keras import layers  # Layer classes (Dense, Dropout, Embedding, ...)
import matplotlib.pyplot as plt  # Plotting graphs
import numpy as np  # Numerical arrays
from sklearn.metrics import ConfusionMatrixDisplay  # Draws the confusion matrix
from sklearn.metrics import classification_report  # Precision / recall / F1 report
from sklearn.metrics import confusion_matrix  # Computes the confusion matrix
import tensorflow as tf  # TensorFlow (tensors, GradientTape, seeds)

tf.random.set_seed(42)                                                # Reproducibility

## Step 2 – Load data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()  # Load MNIST
print("Train:", x_train.shape, y_train.shape)                           # (60000, 28, 28) (60000,)
print("Test :", x_test.shape, y_test.shape)                             # (10000, 28, 28) (10000,)
print("Pixel range before normalization:", x_train.min(), "-", x_train.max())  # 0 - 255

## Step 3 – Normalize data and one-hot encode labels

In [ ]:
x_train = x_train.astype("float32") / 255.0                         # Scale pixels to [0, 1]
x_test = x_test.astype("float32") / 255.0                           # Same for test
print("Pixel range after normalization:", x_train.min(), "-", x_train.max())  # 0.0 - 1.0

y_train_cat = keras.utils.to_categorical(y_train, num_classes=10)   # One-hot training labels (60000, 10)
y_test_cat = keras.utils.to_categorical(y_test, num_classes=10)     # One-hot test labels (10000, 10)
print("Label", y_train[0], "->", y_train_cat[0])                    # Example of one-hot encoding

## Step 4 – Build the softmax classifier

In [ ]:
model = keras.Sequential([                                          # Sequential model
    keras.Input(shape=(28, 28)),                                    # Input: 28x28 image
    layers.Flatten(),                                               # Flatten to 784 values
    layers.Dense(10, activation="softmax"),                         # Single layer: 10 outputs with SOFTMAX
])
model.summary()                                                     # 784*10 + 10 = 7,850 parameters

## Step 5 – Compile with categorical_crossentropy

In [ ]:
model.compile(                                                      # Configure training
    optimizer=keras.optimizers.SGD(learning_rate=0.1),              # Plain SGD works well for this convex problem
    loss="categorical_crossentropy",                                # Needs one-hot labels
    metrics=["accuracy"],                                           # Track accuracy
)

## Step 6 – Train

In [ ]:
history = model.fit(                                                # Train model
    x_train, y_train_cat,                                           # Images and one-hot labels
    epochs=15,                                                      # Number of epochs
    batch_size=128,                                                 # Batch size
    validation_split=0.1,                                           # 10% validation
    verbose=2,                                                      # One line per epoch
)

In [ ]:
plt.figure(figsize=(12, 4))                                         # Figure
plt.subplot(1, 2, 1)                                                # Left plot
plt.plot(history.history["loss"], label="Train")                    # Training loss
plt.plot(history.history["val_loss"], label="Validation")           # Validation loss
plt.title("Loss")                                                   # Title
plt.xlabel("Epoch")                                                 # X label
plt.legend()                                                        # Legend
plt.subplot(1, 2, 2)                                                # Right plot
plt.plot(history.history["accuracy"], label="Train")                # Training accuracy
plt.plot(history.history["val_accuracy"], label="Validation")       # Validation accuracy
plt.title("Accuracy")                                               # Title
plt.xlabel("Epoch")                                                 # X label
plt.legend()                                                        # Legend
plt.tight_layout()                                                  # Tidy
plt.show()                                                          # Show

## Step 7 – Evaluate accuracy on test set

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)  # Evaluate on test data
print(f"Test loss    : {test_loss:.4f}")                             # Print loss
print(f"Test accuracy: {test_acc:.4f}")                              # Print accuracy (~92%)

probs = model.predict(x_test, verbose=0)                             # Softmax probabilities (10000, 10)
y_pred = np.argmax(probs, axis=1)                                    # Predicted class = argmax
print("Probabilities of first test image:", np.round(probs[0], 3))   # Shows softmax output
print("Sum of probabilities:", probs[0].sum())                       # Softmax outputs sum to 1
print("Predicted:", y_pred[0], " True:", y_test[0])                  # Compare prediction to truth

## Step 8 – Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)                                # 10x10 matrix: rows=true, cols=predicted

fig, ax = plt.subplots(figsize=(9, 9))                               # Figure
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax,     # Draw matrix
                                                         cmap="Blues",  # Blue colour map
                                                         colorbar=False)  # No colour bar
plt.title("Confusion Matrix – Softmax Classifier on MNIST")          # Title
plt.show()                                                           # Show

per_class_acc = cm.diagonal() / cm.sum(axis=1)                       # Correct / total for each true class
for digit, acc in enumerate(per_class_acc):                          # Print per-class accuracy
    print(f"Digit {digit}: {acc:.3f}")                               # e.g. digit 5 and 8 are usually hardest

In [ ]:
print(classification_report(y_test, y_pred, digits=4))               # Precision, recall, F1-score per class

## Step 9 – Visualize misclassified images

In [ ]:
wrong = np.where(y_pred != y_test)[0]                                # Indices of wrong predictions
print("Number of misclassified test images:", len(wrong))            # Count errors

plt.figure(figsize=(15, 6))                                          # Figure
for i, idx in enumerate(wrong[:15]):                                 # First 15 mistakes
    plt.subplot(3, 5, i + 1)                                         # 3x5 grid
    plt.imshow(x_test[idx], cmap="gray")                             # Show image
    plt.title(f"True {y_test[idx]}  Pred {y_pred[idx]}", color="red")  # True vs predicted
    plt.axis("off")                                                  # Hide axes
plt.tight_layout()                                                   # Tidy
plt.show()                                                           # Show

## Step 10 – Visualize learned weights (one template per digit)

In [ ]:
W, b = model.layers[-1].get_weights()                                # W shape (784, 10), b shape (10,)
plt.figure(figsize=(15, 3))                                          # Figure
for digit in range(10):                                              # One image per class
    plt.subplot(1, 10, digit + 1)                                    # 1x10 grid
    plt.imshow(W[:, digit].reshape(28, 28), cmap="seismic")          # Positive weights red, negative blue
    plt.title(str(digit))                                            # Digit label
    plt.axis("off")                                                  # Hide axes
plt.suptitle("Weights of each output neuron (what the classifier 'looks for')")  # Title
plt.show()                                                           # Show

## Expected observations
- Test accuracy ≈ **92%** (a single linear layer cannot learn very complex shapes; an MLP reaches ~98%).
- Confusion matrix: most errors between similar digits like 4↔9, 3↔5, 7↔9, 2↔8.
- Weight images look like blurry "templates" of each digit.

## Viva questions
1. Why softmax in the last layer? → Converts scores into a probability distribution across 10 classes.
2. `categorical_crossentropy` vs `sparse_categorical_crossentropy`? → One-hot labels vs integer labels.
3. What does the diagonal of the confusion matrix represent? → Correctly classified samples of each class.
4. How many parameters does this model have? → 784×10 + 10 = 7,850.
5. Why normalize inputs? → Keeps inputs on a similar small scale → better-conditioned gradients, faster convergence.